# 16 — Plant P1: M30 and M35 from the data-entry workbook → experiments for testing

**Input.** `Concrete_Data_Entry_Template_R1.xlsx`, sheet `DATA`, rows for **M-30 (TM-104)** and
**M-35 (TM-20)** of plant **P1** (OPC 53, Panchgaon coarse aggregate, Wainganga natural sand
blended with crushed sand, Fosroc superplasticiser + ATPL Bipolar). Order inputs are set the
way notebook 08 sets them for M30: grade, exposure, RCC, slump, msa, cement type, SCMs.

**Output.** For each grade, the IS 10262 cl 5.8 trial set built around the plant's own recorded
recipe (which is Trial Mix No. 1 — already cast and tested), lab batch sheets for 0.050 m³,
and for M30 the single best next experiment. Every number is rounded to 3 decimals.

**Route.** The SDL's learned models were fitted on a PPC / 100 % M-sand plant. P1 uses OPC 53
and ~70 % natural sand, so its mixes are outside what those models have seen. The project's
fail-safe rule applies: *revert to a verified deterministic design procedure*. Here that is
Layer 0 (IS 10262 + IS 456, `layer0.py`) anchored on the plant's own measured result —
IS 10262 cl 5.1: *"the relationship between strength and free water-cement ratio should
preferably be established for the materials actually to be used"*.

Tags: `[IS]` from the standard · `[EI]` engineering interpretation · `[PC]` project choice ·
`[ASSUMED]` not in the data, declared · `[DATA-FIX]` a data-entry error in the workbook, corrected
and shown.

In [ ]:
import sys as _sys
for _m in ("is_rules", "layer0", "workability_residual", "strength_residual",
           "screen", "selection", "loop"):
    _sys.modules.pop(_m, None)
import math, json
from dataclasses import replace
from pathlib import Path
import numpy as np
import pandas as pd
import joblib, openpyxl
import is_rules as IS
import layer0 as L0
import strength_residual as SR
import screen as SC
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
R3 = lambda x: None if x is None or (isinstance(x, float) and math.isnan(x)) else round(float(x), 3)

## 1. Read the M30 and M35 rows exactly as entered

In [ ]:
SRC = "Concrete_Data_Entry_Template_R1.xlsx"
ws = openpyxl.load_workbook(SRC, data_only=True)["DATA"]
rows = list(ws.iter_rows(values_only=True))
HDR = rows[1]
recs = [dict(zip(HDR, r)) for r in rows[5:] if r and any(v not in (None, "") for v in r)]
RAW = {str(r["grade"]).replace("-", "").replace(" ", "").upper(): r for r in recs if r.get("grade")}
print("grades in the workbook:", sorted(RAW))
SHOW = ["source_row", "plant_lab_id", "batch_volume", "msa", "cement", "flyash", "water_batched",
        "water_basis", "agg_20mm", "agg_10mm", "crushed_sand", "natural_sand", "admixture1",
        "admixture2", "ca_moisture", "sand_moisture", "slump_45min", "slump_60min", "cube_7d_1",
        "cube_7d_2", "cube_7d_3", "cube_28d_1", "cube_28d_2", "cube_28d_3", "target_mean_strength",
        "exposure", "min_cement_specified", "max_wc_specified", "cement_type", "cement_28d_strength",
        "ca_sg", "ca_absorption", "sand_sg", "sand_absorption", "sand_zone", "sand_pass_600"]
print(pd.DataFrame({g: {k: RAW[g].get(k) for k in SHOW} for g in ("M30", "M35")}).to_string())

## 2. Data-entry checks — what the workbook says vs what the batch must have been

The rows were entered from 6 m³ plant batch sheets. Several columns are shifted or reused.
Each fix below is tested, not assumed: a reading is accepted only if the absolute volumes
then add up to 1 m³ (IS 10262 cl 5.7 method; `[PC]` tolerance 0.02 m³ because batched
masses carry moisture).

In [ ]:
AIR = IS.IS10262_TABLE3_AIR[20] / 100.0          # [IS] Table 3, 20 mm
FA_SG = 2.20                                      # [ASSUMED] fly ash SG not in the workbook
ADM_SG = 1.10                                     # [ASSUMED] admixture SG not in the workbook
FIX = []                                          # (grade, field, as entered, used, why)


def reconstruct(g):
    r = RAW[g]
    V = float(r["batch_volume"])
    ca_w = sum(float(r.get(k) or 0) for k in ("agg_40mm", "agg_20mm", "agg_10mm", "agg_other")) / V
    cs_w, ns_w = float(r["crushed_sand"] or 0) / V, float(r["natural_sand"] or 0) / V
    fs_w = cs_w + ns_w
    if abs(float(r["msa"]) / V - ca_w) < 0.5:
        FIX.append((g, "msa", r["msa"], 20.0, "[DATA-FIX] the msa column holds the total coarse aggregate "
                    "of the batch; 20 mm + 10 mm fractions -> msa 20 mm"))
    fa = float(r["flyash"] or 0) / V
    water_add = float(r["water_batched"]) / V
    adm = (float(r["admixture1"] or 0) + float(r["admixture2"] or 0)) / V
    ms, mc = float(r["sand_moisture"]), float(r["ca_moisture"])
    # [EI] template: 'free surface moisture'. Aggregates are batched wet: weighed = SSD (1 + m/100).
    sand_ssd, ca_ssd = fs_w / (1 + ms / 100), ca_w / (1 + mc / 100)
    free_water = water_add + (fs_w - sand_ssd) + (ca_w - ca_ssd)

    def closure(cem, ca_sg):
        return (cem / 3150 + fa / (FA_SG * 1000) + free_water / 1000 + adm / (ADM_SG * 1000)
                + ca_ssd / (ca_sg * 1000) + sand_ssd / (float(r["sand_sg"]) * 1000) + AIR)

    cem_entered = float(r["cement"]) / V
    readings = {"cement column / batch volume": cem_entered,
                "min_cement_specified (cement)": float(r["min_cement_specified"]),
                "min_cement_specified minus fly ash (cementitious)": float(r["min_cement_specified"]) - fa}
    ca_sgs = {"as entered": float(r["ca_sg"]), "M30 row value 2.87": 2.87}
    table = []
    for rk, cem in readings.items():
        for sk, sg in ca_sgs.items():
            table.append((rk, sk, cem, sg, closure(cem, sg)))
    T = pd.DataFrame(table, columns=["cement reading", "CA SG", "cement kg/m3", "ca_sg", "volume m3"])
    T["off_by"] = (T["volume m3"] - 1).abs()
    best = T.sort_values("off_by").iloc[0]
    if abs(cem_entered - fs_w) < 0.5:
        FIX.append((g, "cement", r["cement"], R3(best["cement kg/m3"] * V),
                    f"[DATA-FIX] the cement column equals the total sand of the batch ({fs_w * V:.1f}); "
                    f"cement taken from the reading whose volumes close: '{best['cement reading']}'"))
    if best["ca_sg"] != float(r["ca_sg"]):
        FIX.append((g, "ca_sg", r["ca_sg"], best["ca_sg"], "[DATA-FIX] entered CA SG equals the sand SG; "
                    "2.87 (same quarry, M30 row) closes the volumes"))
    if r.get("max_wc_specified") and float(r["max_wc_specified"]) > 1:
        FIX.append((g, "max_wc_specified", r["max_wc_specified"], 0.45,
                    "[DATA-FIX] 450 is the IS 456 cl 8.2.4.2 cement ceiling, not a w/c; max w/c for severe "
                    "RCC is 0.45 (IS 456 Table 5)"))
    cubes28 = [float(r[f"cube_28d_{i}"]) for i in (1, 2, 3)]
    cubes7 = [float(r[f"cube_7d_{i}"]) for i in (1, 2, 3)]
    c28 = IS.cube_set_result(cubes28)
    print(f"\n{g}: volume closure for every reading (target 1.000 m3 incl. {AIR:.2f} air)")
    print(T.round(3).to_string(index=False))
    return {"grade": g, "V": V, "cement": float(best["cement kg/m3"]), "flyash": fa,
            "free_water": free_water, "water_added": water_add, "admixture": adm,
            "ca_ssd": ca_ssd, "sand_ssd": sand_ssd, "ca_sg": float(best["ca_sg"]),
            "sand_sg": float(r["sand_sg"]), "crushed_frac": cs_w / fs_w,
            "ca_split_20_10": (float(r["agg_20mm"]), float(r["agg_10mm"])),
            "sand_free_moist": ms, "ca_free_moist": mc,
            "sand_abs": float(r["sand_absorption"]), "ca_abs": float(r["ca_absorption"]),
            "closure": float(best["volume m3"]), "fc28": c28["result_mpa"], "fc28_raw_mean": sum(cubes28) / 3,
            "fc28_check": c28["outcome"], "fc7": sum(cubes7) / 3,
            "slump45": float(r["slump_45min"]), "slump60": float(r["slump_60min"]),
            "temp": float(r["concrete_temp"]), "cement_28d": float(r["cement_28d_strength"]),
            "target_mean_entered": float(r["target_mean_strength"]), "pass600": float(r["sand_pass_600"]),
            "source_row": r["source_row"]}


P = {g: reconstruct(g) for g in ("M30", "M35")}
for g in P:
    p = P[g]
    p["binder"] = p["cement"] + p["flyash"]
    p["wcm"] = p["free_water"] / p["binder"]
    p["fa_frac"] = p["flyash"] / p["binder"]
    p["adm_pct"] = 100 * p["admixture"] / p["binder"]
print("\nDATA FIXES")
for f in FIX:
    print(f"  {f[0]}  {f[1]:18s} entered {f[2]!s:>8s} -> used {f[3]!s:>8s}   {f[4]}")
print("\nPER m3 (SSD aggregates, free water)")
print(pd.DataFrame(P).T[["cement", "flyash", "binder", "free_water", "admixture", "ca_ssd", "sand_ssd",
                         "wcm", "fa_frac", "adm_pct", "closure", "fc28", "fc7", "slump45"]]
      .astype(float).round(3).to_string())

## 3. The order inputs — set the way notebook 08 sets them for M30

| input | M30 | M35 | classification |
|---|---|---|---|
| grade | M30 | M35 | ENGINEER INPUT (workbook) |
| exposure | severe | severe | **[ASSUMED]** blank in the workbook; notebook 08 uses severe for M30 |
| RCC | yes | yes | ENGINEER INPUT (workbook) |
| target slump | 150 mm | 180 mm | **[ASSUMED]** blank; the plant's own slump at 45 min is taken as the delivery requirement |
| msa | 20 mm | 20 mm | MATERIAL-SPECIFIC (20 + 10 mm fractions) |
| placing | pumped (boom) | pumped | ENGINEER INPUT |
| cement | OPC 53, 55 MPa | OPC 53, 57 MPa | MATERIAL-SPECIFIC (certificate week) |
| fly ash | Grade I, SG 2.20 | not used | SG **[ASSUMED]** |
| admixture | Fosroc SP + ATPL Bipolar | same | SG 1.10 **[ASSUMED]**; solids unknown -> whole dose counted as water `[EI]` |

In [ ]:
SLUMP = {"M30": P["M30"]["slump45"], "M35": P["M35"]["slump45"]}
EVID = {"M30": ("P1 batch records in Concrete_Data_Entry_Template_R1.xlsx: TM-104 (M30, 20 % fly ash, "
                "28 d 38.0) and TM-18 (M50, 13 % fly ash, 28 d 66.5) -- 2 batches. ENGINEER TO CONFIRM "
                "(IS 456 Table 5 Note 1 'if the suitability is established')"), "M35": None}
# [PLANT-OBSERVED] admixture dose range, % of binder, across every P1 batch in the workbook.
#  IS 9103 cl 10.2(f): the manufacturer declares the recommended dosage -- replace with the
#  Fosroc / ATPL datasheet range when it is available.
_d = []
for _g, _r in RAW.items():
    try:
        _V = float(_r["batch_volume"])
        _adm = (float(_r["admixture1"] or 0) + float(_r["admixture2"] or 0)) / _V
        _cem = P[_g]["cement"] if _g in P else float(_r["cement"]) / _V
        _d.append(100 * _adm / (_cem + float(_r["flyash"] or 0) / _V))
    except Exception:
        pass
ADM_RANGE = (round(min(_d), 3), round(max(_d), 3))
print(f"admixture dose range observed at P1: {ADM_RANGE} % of binder [PLANT-OBSERVED, not the IS 9103 declaration]")
STATE, REQ = {}, {}
for g, p in P.items():
    STATE[g] = L0.MaterialState(
        cement_type="OPC", cement_grade=53, cement_sg=3.15, cement_28d_strength=p["cement_28d"],
        flyash_sg=FA_SG if p["flyash"] > 0 else None, sand_type="natural",
        sand_sg=p["sand_sg"], sand_absorption_pct=p["sand_abs"], sand_passing_600um=p["pass600"],
        ca_sg=p["ca_sg"], ca_shape="angular", ca_absorption_pct=p["ca_abs"], msa_mm=20.0,
        scms_available=("flyash",), admixture_sg=ADM_SG, admixture_solids_pct=None,
        admixture_dosage_range_pct=ADM_RANGE,
        combination_suitability_evidence=EVID[g], ca_split_20_10=p["ca_split_20_10"])
    z = STATE[g].sand_zone()
    print(f"{g}: sand 600 um {p['pass600']} % -> Zone {z['zone']} ({z['source']}); "
          f"target mean {IS.target_mean_strength(g)['target_mean']} (IS 10262 cl 4.2, S = 5 assumed, Table 2; "
          f"workbook says {p['target_mean_entered']})")

## 4. Can the SDL's learned models be used for P1? (domain check)

In [ ]:
wmodel = joblib.load("workability_residual_model.joblib")
smodel = SC.load_strength_model()
train = pd.read_csv("04_workability_dataset.csv")
train = train[train["design_grp"].astype(str).str.contains("[A-Za-z]", na=False)]
SUP = SC.training_support(train, wmodel["features"])
for g, p in P.items():
    why = []
    if smodel["cement_type"] != "OPC53":
        try:
            SR.fig1_strength(0.45, cement_type="OPC53")
        except SR.CurveNotDigitised as e:
            why.append(f"strength: model fitted on {smodel['cement_type']}; IS 10262 Fig 1 Curve 3 (53 MPa "
                       f"cement) is not digitised -- {str(e)[:80]}...")
    lo, hi = SUP["msand_frac"]
    if not (lo <= p["crushed_frac"] <= hi):
        why.append(f"workability: crushed-sand share {p['crushed_frac']:.3f} outside training [{lo:.3f}, {hi:.3f}]")
    sl = train["slump_mm"].astype(float)
    if SLUMP[g] > sl.quantile(0.95):
        why.append(f"workability: target slump {SLUMP[g]:.0f} mm above 95 % of training slumps ({sl.quantile(0.95):.0f})")
    print(f"{g}: " + ("INSIDE" if not why else "OUTSIDE the models' domain"))
    for w in why:
        print("   - " + w)
print("\n=> fail-safe: Layer 0 (IS 10262 / IS 456) anchored on the plant's own measured batch. "
      "No ML prediction is used to accept a mix.")

## 5. The plant's own recipe through the IS gate (this is Trial Mix No. 1 — already cast)

In [ ]:
for g, p in P.items():
    gate = IS.compliance_gate(
        cement_kg_m3=p["cement"], fly_ash_kg_m3=p["flyash"], ggbs_kg_m3=0, other_scm_kg_m3=0,
        grade=g, exposure=IS.Exposure.SEVERE, use=IS.UseClass.RCC, msa_mm=20,
        free_water_kg_m3=p["free_water"], cement_type="OPC", volume_closure_m3=p["closure"],
        admixture_kg_m3=p["admixture"], admixture_dosage_range_pct=STATE[g].admixture_dosage_range_pct,
        admixture_water_kg_m3=p["admixture"], combination_suitability_evidence=EVID[g])
    # The volume-closure check is for DESIGNS. A batched record carries moisture and weighing
    # error, so its closure (section 2) is reported, not used to judge the recipe.
    hard_fail = [c["check"] for c in gate["checks"] if c["outcome"] in ("fail", "override_required")
                 and c["check"] != "volume closure"]
    p["gate_decided"] = not hard_fail
    print(f"\n{g} plant recipe {p['source_row']}: IS checks (closure excluded) "
          f"{'PASS' if not hard_fail else 'FAIL ' + str(hard_fail)}; 28-d {p['fc28']:.3f} "
          f"vs target mean {IS.target_mean_strength(g)['target_mean']:.3f} "
          f"({'MEETS' if p['fc28'] >= IS.target_mean_strength(g)['target_mean'] else 'BELOW'})")
    for c in gate["checks"]:
        if c["outcome"] != "pass" or c["check"] in ("maximum free w/c", "minimum binder", "maximum cement"):
            print(f"   [{c['outcome']:>17s}] {c['check']:42s} {c['detail'][:120]}")
    if p["flyash"] > 0:
        no_ev = IS.compliance_gate(
            cement_kg_m3=p["cement"], fly_ash_kg_m3=p["flyash"], ggbs_kg_m3=0, other_scm_kg_m3=0,
            grade=g, exposure=IS.Exposure.SEVERE, use=IS.UseClass.RCC, msa_mm=20,
            free_water_kg_m3=p["free_water"], cement_type="OPC", volume_closure_m3=p["closure"],
            admixture_water_kg_m3=p["admixture"])
        wc = [c for c in no_ev["checks"] if c["check"] == "maximum free w/c"][0]
        print(f"   WITHOUT fly-ash suitability recorded: max free w/c -> {wc['outcome']} ({wc['detail'][:90]})")

## 6. IS 10262 cl 5.8 trial set around the plant recipe

Same water as the plant's batch in every trial (cl 5.8), w/cm about −10 % and +10 %; the
fly-ash share and the admixture percent are held (`[PC]`). The coarse-aggregate volume is the
plant's own, which sits inside the band IS 10262 Table 5 + cl 5.5.1 + cl 5.5.2 permits; it
then moves with w/cm by cl 5.5.1.

In [ ]:
TRIALS, BASE = {}, {}
for g, p in P.items():
    st = STATE[g]
    gross = IS.water_content(g, 20, SLUMP[g], ca_shape="angular")["water_kg_m3"]
    red = 100.0 * (1.0 - p["free_water"] / gross)      # the plant's water as a cl 5.3 reduction
    rq0 = L0.Requirement(grade=g, target_slump_mm=SLUMP[g], exposure=IS.Exposure.SEVERE,
                         use=IS.UseClass.RCC, pumpable=True, pumpable_reduction_frac=0.0,
                         water_reduction_pct=red, concrete_temp_c=p["temp"])
    band = L0.permitted_fine_fraction_band(st, rq0, p["wcm"])
    ca_plant = (p["ca_ssd"] / p["ca_sg"]) / (p["ca_ssd"] / p["ca_sg"] + p["sand_ssd"] / p["sand_sg"])
    f = (band["ca_volume_nominal"] - ca_plant) / (band["ca_volume_nominal"] - band["ca_volume_min"])
    inside = 0.0 <= f <= 1.0
    f = min(max(f, 0.0), 1.0)
    print(f"{g}: plant CA volume fraction {ca_plant:.3f}; IS band at w/cm {p['wcm']:.3f}: "
          f"{band['ca_volume_min']:.3f} (cl 5.5.2 full) to {band['ca_volume_nominal']:.3f} (Table 5 + cl 5.5.1) "
          f"-> {'inside' if inside else 'OUTSIDE, clamped'}; uses {f:.3f} of the cl 5.5.2 allowance")
    rq = replace(rq0, pumpable_reduction_frac=f)
    REQ[g] = rq
    BASE[g] = L0.design_mix(rq, st, binder_kg_m3=p["binder"], flyash_frac=p["fa_frac"],
                            admixture_pct_of_binder=p["adm_pct"])
    b = BASE[g]
    print(f"   Layer 0 rebuild of the plant recipe: cement {b.cement_kg_m3:.3f}, fly ash {b.flyash_kg_m3:.3f}, "
          f"water {b.water_kg_m3:.3f}, CA {b.coarse_agg_kg_m3:.3f} (plant {p['ca_ssd']:.3f}), "
          f"FA {b.fine_agg_kg_m3:.3f} (plant {p['sand_ssd']:.3f}), decided {b.decided}")
    TRIALS[g] = L0.trial_set(rq, st, binder_kg_m3=p["binder"], flyash_frac=p["fa_frac"],
                             admixture_pct_of_binder=p["adm_pct"])
    for t in TRIALS[g]:
        m = t["mix"]
        print(f"   trial {t['trial_no']}: " + ("REFUSED " + t["note"] if m is None else
              f"binder {m.binder_kg_m3:.3f}  w/cm {m.wc_ratio:.3f}  w/c counted {m.wc_counted:.3f}  "
              f"decided {m.decided}  -- {t['note'][:110]}"))

## 7. What strength to expect — anchored on the plant's measured batch

`[EI]` For each grade the strength–w/cm curve is passed through the plant's own measured
28-day result (IS 10262 cl 5.1: use the relationship for the materials actually used). Its
slope is taken two ways and both are shown: from the IS 10262 Fig 1 curve digitised in
notebook 10b, and fitted to P1's own four OPC 53 batches (M30, M35, M45, M50) in the
workbook. These are **expectations for the lab, not acceptance values**.

In [ ]:
b_IS = -math.log(SR.fig1_strength(0.50, "PPC") / SR.fig1_strength(0.40, "PPC")) / 0.10

pts = []
for g, r in RAW.items():
    try:
        V = float(r["batch_volume"])
        ca_w = sum(float(r.get(k) or 0) for k in ("agg_20mm", "agg_10mm")) / V
        fs_w = (float(r["crushed_sand"] or 0) + float(r["natural_sand"] or 0)) / V
        ms, mc = float(r["sand_moisture"]), float(r["ca_moisture"])
        fw = float(r["water_batched"]) / V + fs_w * ms / (100 + ms) + ca_w * mc / (100 + mc)
        cem = P[g]["cement"] if g in P else float(r["cement"]) / V
        binder = cem + float(r["flyash"] or 0) / V
        fc = sum(float(r[f"cube_28d_{i}"]) for i in (1, 2, 3)) / 3
        pts.append((g, fw / binder, fc, mc < -float(r["ca_absorption"])))
    except Exception as e:
        print(f"{g}: skipped ({e})")
PT = pd.DataFrame(pts, columns=["grade", "wcm", "fc28", "ca_moisture_below_dry"])
print(PT.round(3).to_string(index=False))
ok = PT[~PT["ca_moisture_below_dry"]]
b_plant = -np.polyfit(ok["wcm"], np.log(ok["fc28"]), 1)[0]
print(f"\nslope d ln(fc)/d(w/cm): IS 10262 Fig 1 (digitised) {b_IS:.3f};  P1 fit on {len(ok)} batches {b_plant:.3f}"
      f"  (rows whose CA moisture is below oven-dry are excluded -- physically impossible)")


def expected_fc(g, wcm):
    p = P[g]
    return {k: p["fc28"] * math.exp(-b * (wcm - p["wcm"])) for k, b in (("IS_slope", b_IS), ("P1_slope", b_plant))}

## 8. Trials table and lab batch sheets (0.050 m³, 3 decimals)

In [ ]:
VOL = IS.trial_batch_volume(6, msa_mm=20)["batch_m3"]
rows_t, rows_b = [], []
for g, ts in TRIALS.items():
    p, st = P[g], STATE[g]
    tm = IS.target_mean_strength(g)["target_mean"]
    s_tot = IS.total_moisture_from_surface(p["sand_abs"], p["sand_free_moist"])
    c_tot = IS.total_moisture_from_surface(p["ca_abs"], p["ca_free_moist"])
    for t in ts:
        m = t["mix"]
        if m is None:
            continue
        e = expected_fc(g, m.wc_ratio)
        status = "CAST -- plant batch " + str(p["source_row"]) if t["trial_no"] == 1 else "TO CAST"
        rows_t.append({
            "grade": g, "trial_no": t["trial_no"], "status": status, "what": t["note"],
            "cement_kg_m3": m.cement_kg_m3, "flyash_kg_m3": m.flyash_kg_m3, "binder_kg_m3": m.binder_kg_m3,
            "free_water_kg_m3": m.water_kg_m3, "admixture_kg_m3": m.admixture_kg_m3,
            "admixture_pct_binder": 100 * m.admixture_kg_m3 / m.binder_kg_m3,
            "coarse_20_10_ssd_kg_m3": m.coarse_agg_kg_m3 * p["ca_split_20_10"][0] / sum(p["ca_split_20_10"]),
            "coarse_10_475_ssd_kg_m3": m.coarse_agg_kg_m3 * p["ca_split_20_10"][1] / sum(p["ca_split_20_10"]),
            "crushed_sand_ssd_kg_m3": m.fine_agg_kg_m3 * p["crushed_frac"],
            "natural_sand_ssd_kg_m3": m.fine_agg_kg_m3 * (1 - p["crushed_frac"]),
            "wcm_total": m.wc_ratio, "wc_counted_IS456": m.wc_counted,
            "ca_volume_fraction": m.ca_volume_fraction, "volume_closure_m3": m.volume_closure_m3,
            "gate_decided": m.decided, "target_mean_mpa": tm,
            "expected_28d_IS_slope": e["IS_slope"], "expected_28d_P1_slope": e["P1_slope"],
            "measured_28d": p["fc28"] if t["trial_no"] == 1 else None,
            "measured_7d": p["fc7"] if t["trial_no"] == 1 else None,
            "measured_slump_45min": p["slump45"] if t["trial_no"] == 1 else None})
        bs = L0.batch_sheet(m, st, volume_m3=VOL, sand_moisture_pct=s_tot, ca_moisture_pct=c_tot,
                            round_kg=0.001)
        a20, a10 = p["ca_split_20_10"]
        exact = {"cement": m.cement_kg_m3, "fly ash": m.flyash_kg_m3, "GGBS": m.ggbs_kg_m3,
                 "water": m.water_kg_m3, "admixture": m.admixture_kg_m3, "fine": m.fine_agg_kg_m3,
                 "coarse 20-10": m.coarse_agg_kg_m3 * a20 / (a20 + a10),
                 "coarse 10-4.75": m.coarse_agg_kg_m3 * a10 / (a20 + a10),
                 "coarse aggregate": m.coarse_agg_kg_m3}
        for r_ in bs["rows"]:
            # full-precision design value (batch_sheet rounds its echo to 2 decimals)
            r_["design_ssd_kg_m3"] = next(v for k, v in exact.items() if r_["material"].startswith(k))
            if r_["material"].startswith("fine"):
                for nm, fr in (("crushed sand (as weighed)", p["crushed_frac"]),
                               ("natural sand (as weighed)", 1 - p["crushed_frac"])):
                    rows_b.append({"grade": g, "trial_no": t["trial_no"], "material": nm,
                                   "design_ssd_kg_m3": r_["design_ssd_kg_m3"] * fr,
                                   "batch_kg": r_["batch_kg"] * fr})
            else:
                rows_b.append({"grade": g, "trial_no": t["trial_no"], **r_})
        rows_b.append({"grade": g, "trial_no": t["trial_no"], "material": "batch volume m3 / moisture used",
                       "design_ssd_kg_m3": None, "batch_kg": None,
                       "note": f"{VOL} m3; sand total moisture {s_tot:.3f} %, CA {c_tot:.3f} % of dry mass "
                               f"(plant free moisture {p['sand_free_moist']} / {p['ca_free_moist']} % converted, "
                               f"IS 2386 Pt 3 cl 5.5.1) -- MEASURE ON THE DAY"})
TRI = pd.DataFrame(rows_t)
BAT = pd.DataFrame(rows_b)
num = TRI.select_dtypes("number").columns
TRI[num] = TRI[num].round(3)
BAT[["design_ssd_kg_m3", "batch_kg"]] = BAT[["design_ssd_kg_m3", "batch_kg"]].astype(float).round(3)
print(TRI[["grade", "trial_no", "status", "cement_kg_m3", "flyash_kg_m3", "free_water_kg_m3",
           "admixture_kg_m3", "wcm_total", "wc_counted_IS456", "expected_28d_IS_slope",
           "expected_28d_P1_slope", "target_mean_mpa", "measured_28d"]].to_string(index=False))

## 9. The best next experiment per grade — and the single best for M30 (industrial view)

Trial Mix No. 1 for M30 is the plant's own batch TM-104: **already cast**, 28-day 38.0 MPa —
just **below** the 38.25 MPa target mean (IS 10262 cl 4.2). A plant QC engineer with one more
batch to spend wants a batch that (1) passes the IS gate, (2) is expected to clear the target
mean under **both** slope estimates, so it is not a wasted cube set, and (3) together with
TM-104 gives the plant's own strength–w/cm line, so the production w/cm can be read off
without further trials (IS 10262 Annex A-15). Among the batches that meet (1) and (2), the one
with the least cementitious material is chosen.

In [ ]:
cols = ["cement_kg_m3", "flyash_kg_m3", "free_water_kg_m3", "admixture_kg_m3", "coarse_20_10_ssd_kg_m3",
        "coarse_10_475_ssd_kg_m3", "crushed_sand_ssd_kg_m3", "natural_sand_ssd_kg_m3", "binder_kg_m3",
        "wcm_total", "wc_counted_IS456", "expected_28d_IS_slope", "expected_28d_P1_slope", "target_mean_mpa"]
BESTS = []
for g in ("M30", "M35"):
    M = TRI[(TRI.grade == g) & (TRI.status == "TO CAST")].copy()
    M["clears_target_both_slopes"] = (M[["expected_28d_IS_slope", "expected_28d_P1_slope"]].min(axis=1)
                                      >= M["target_mean_mpa"])
    print(f"\n{g} candidates")
    print(M[["trial_no", "binder_kg_m3", "wcm_total", "wc_counted_IS456", "gate_decided",
             "expected_28d_IS_slope", "expected_28d_P1_slope", "target_mean_mpa",
             "clears_target_both_slopes"]].to_string(index=False))
    ok = M[M.gate_decided & M.clears_target_both_slopes]
    if ok.empty:
        print(f"{g}: no trial is both IS-decided and expected to clear the target mean -- none chosen")
        continue
    b = ok.sort_values("binder_kg_m3").iloc[0].copy()
    # what production w/cm the measured line would point to if this trial lands as expected [EI]
    p = P[g]; tm = IS.target_mean_strength(g)["target_mean"]
    for k, bb in (("IS", b_IS), ("P1", b_plant)):
        x = p["wcm"] - math.log(tm / p["fc28"]) / bb
        b[f"implied_production_wcm_{k}_slope"] = x
        b[f"implied_production_binder_{k}_slope"] = p["free_water"] / x
    BESTS.append(b)
    print(f"\nBEST NEXT {g} EXPERIMENT: Trial Mix No. {int(b.trial_no)}  ({b.what})")
    for c in cols + [c for c in b.index if c.startswith("implied_")]:
        print(f"   {c:36s} {float(b[c]):10.3f}")
BEST = [b for b in BESTS if b.grade == "M30"][0]

## 10. Write the outputs

In [ ]:
TRI.to_csv("16_P1_M30_M35_trials.csv", index=False)
BAT.to_csv("16_P1_batch_sheets.csv", index=False)
pd.DataFrame([BEST]).round(3).to_csv("16_P1_M30_best_next_experiment.csv", index=False)
pd.DataFrame(BESTS).round(3).to_csv("16_P1_best_next_experiments_M30_M35.csv", index=False)
pd.DataFrame(FIX, columns=["grade", "field", "as_entered", "used", "why"]).to_csv("16_P1_data_fixes.csv", index=False)

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
wb = Workbook(); GREY = PatternFill("solid", fgColor="E7E6E6"); B = Font(bold=True)
def sheet(name, df, note=None, first=False):
    w = wb.active if first else wb.create_sheet(name)
    w.title = name
    r = 1
    if note:
        w.cell(row=1, column=1, value=note).font = B; r = 3
    for j, c in enumerate(df.columns, 1):
        x = w.cell(row=r, column=j, value=str(c)); x.font = B; x.fill = GREY; x.alignment = Alignment(wrap_text=True)
        w.column_dimensions[x.column_letter].width = 16
    for i, row in enumerate(df.itertuples(index=False), r + 1):
        for j, v in enumerate(row, 1):
            v = None if (isinstance(v, float) and math.isnan(v)) else v
            c = w.cell(row=i, column=j, value=v)
            if isinstance(v, float):
                c.number_format = "0.000"
_bt = pd.DataFrame(BESTS).set_index("grade").T.reset_index().rename(columns={"index": "item"})
_bt = _bt.apply(lambda c: c.map(lambda v: round(v, 3) if isinstance(v, float) else v))
sheet("Best next experiments", _bt,
      "Best next experiment per grade -- M30 column is the single best M30 batch (COMPUTATIONALLY COMPLIANT; not yet cast)",
      first=True)
sheet("Trials", TRI, "IS 10262 cl 5.8 trial sets -- plant P1, OPC 53; trial 1 = plant batch already cast")
sheet("Batch sheets", BAT, f"Lab batch {VOL} m3 -- weights to 3 decimals; measure aggregate moisture on the day")
sheet("Data fixes", pd.DataFrame(FIX, columns=["grade", "field", "as_entered", "used", "why"]),
      "Data-entry corrections found in Concrete_Data_Entry_Template_R1.xlsx")
wb.save("16_P1_M30_M35_trial_pack.xlsx")
for f in ("16_P1_M30_M35_trials.csv", "16_P1_batch_sheets.csv", "16_P1_M30_best_next_experiment.csv",
          "16_P1_best_next_experiments_M30_M35.csv",
          "16_P1_data_fixes.csv", "16_P1_M30_M35_trial_pack.xlsx"):
    print("wrote", f)